In [ ]:
# For more info on this exercise, see https://medium.com/mlearning-ai/topic-modelling-with-lda-on-the-tweets-mentioning-elon-musk-687076a2c86b
# Import libraries
import tweepy
import pandas as pd
import csv
import requests
import io
import json

import warnings
warnings.filterwarnings("ignore") # Ignore warning messages

In [ ]:
# Import libraries for text preprocessing

import nltk
nltk.download('all')

from nltk.tag import pos_tag

from nltk.sentiment.vader import SentimentIntensityAnalyzer

from nltk.corpus import stopwords

from nltk.tokenize import word_tokenize

from nltk.stem import WordNetLemmatizer

In [ ]:
# Import libraries for LDA
# Need to first install gensim, spacy, and pyLDAvis through command prompt

import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

#spacy
import spacy

#vis
import pyLDAvis
import pyLDAvis.gensim_models

In [ ]:
# Enter your X/Twitter credentials here
consumer_key= "removed to prevent unauthorized access, unexpected billing charges, and compromised data security"
secret_key= "removed to prevent unauthorized access, unexpected billing charges, and compromised data security"
bearer_token="removed to prevent unauthorized access, unexpected billing charges, and compromised data security"

In [ ]:
# Create Tweepy connection client
client = tweepy.Client( bearer_token=bearer_token, 
                        consumer_key=consumer_key, 
                        consumer_secret=secret_key, 
                        return_type = requests.Response,
                        wait_on_rate_limit=True)

In [ ]:
# Define query as original tweets with keyword NBA
query = 'ICEgov -is:retweet'

# get max. 100 tweets
tweets = client.search_recent_tweets(query=query, 
                                    tweet_fields=['author_id', 'text','created_at'],  
                                    max_results=100)

In [ ]:
# Save data as dictionary
tweets_dict = tweets.json() 

# Extract "data" value from dictionary
tweets_data = tweets_dict['data'] 

# Transform to pandas Dataframe
tweets_df = pd.json_normalize(tweets_data) 

In [ ]:
tweets_df

In [ ]:
import re, string

# Remove HTML links and @username

def remove_noise(tweet_tokens):

    cleaned_tokens = []

    for token, tag in pos_tag(tweet_tokens):
        token = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+#]|[!*\(\),]|'\
                       '(?:%[0-9a-fA-F][0-9a-fA-F]))+',' ', token)  # Remove URLs
        token = re.sub('(@[A-Za-z0-9_]+)',' ', token)  # Remove @username
        

        # If the token/word is not empty and is not a punctuation mark, then convert taken to lower case
        # and append to cleaned_tokens
        if len(token) > 0 and token not in string.punctuation: 
            cleaned_tokens.append(token.lower())
    return cleaned_tokens

# apply the function to the tweets data frame
tweets_df['text'] = remove_noise(tweets_df['text'].tolist())
tweets_df

In [ ]:
# Remove non-alphabetic characters
def keep_only_alphabet(tweet_tokens):
    return re.sub('[^a-z]', ' ', tweet_tokens)


# Apply function to texts of all tweets
tweets_df.text=tweets_df.text.apply(keep_only_alphabet)
tweets_df

In [ ]:
# create preprocess_text function
def preprocess_text(text):

    # Tokenize the text
    tokens = word_tokenize(text.lower())  # Converts to lower case and then tokenize


    # Remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]

    # Lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]

    # Join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)

    return processed_text

# apply the function to the tweets data frame
tweets_df['text'] = tweets_df['text'].apply(preprocess_text)
tweets_df

In [ ]:
# Convert tweet text into tokens (i.e., words)

def generate_tokens(tweet):
    words=[]
    for word in tweet.split(' '):
    # using the if condition because we introduced extra spaces during text cleaning
      if word!='':
         words.append(word)
    return words

#storing the generated tokens in a new column named 'words'
tweets_df['words']=tweets_df.text.apply(generate_tokens)

tweets_df

In [ ]:
# Create dictionary of unique tokens/words in tweets

def create_dictionary(words):
    return corpora.Dictionary(words)

#passing the dataframe column having tokens as the argument
id2word=create_dictionary(tweets_df.words)

print(id2word)

In [ ]:
# Create document matrix
def create_document_matrix(tokens,id2word):
    corpus = []
    for text in tokens:
       corpus.append(id2word.doc2bow(text)) # convert tweet text into bag of words format of token id and token count
    return corpus

#passing the dataframe column having tokens and dictionary
corpus=create_document_matrix(tweets_df.words,id2word)
print(tweets_df.words[0]) # print out wrods/tokens in first tweet
print(corpus[0]) # print out token id and token count of tokens in first tweet

In [ ]:
# LDA analysis with 10 topics
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
 id2word=id2word,
 num_topics=10,
 random_state=100,
 )

In [ ]:
# Generate LDA topics

def get_lda_topics(model, num_topics, top_n_words):
     word_dict = {}
     for i in range(num_topics):
        # Creates variable with top n words for each topic
         word_dict['Topic # ' + '{:02d}'.format(i+1)] = [i[0] for i in model.show_topic(i, topn = top_n_words)];
 
     return pd.DataFrame(word_dict)
                   
get_lda_topics(lda_model,10,10) # 10 topics and 10 top words for each topic

In [ ]:
# Visualizing the topics
pyLDAvis.enable_notebook() 
# mds specifies methods to claculating distance between topics 
# R speficies num of terms to display
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word, mds='mmds', R=30) 

# for more info on how to interpret the results, see https://neptune.ai/blog/pyldavis-topic-modelling-exploration-tool-that-every-nlp-data-scientist-should-know
vis